<a href="https://colab.research.google.com/github/Sahil-Tamrakar/Computational-Linguistic-Natural-Language-Processing-LAB/blob/main/Lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Experiment-4: Term Frequency and NER**

In [1]:
# Install required libraries
!pip install -q spacy pandas tabulate
!python -m spacy download en_core_web_sm -q

import re
import math
import string
import pandas as pd
import spacy
from collections import Counter
from google.colab import files

# Load spaCy NLP model
nlp = spacy.load("en_core_web_sm")

print("Setup completed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 91.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Setup completed successfully!


Experiment 4.1: TF Analysis & NER (Using Toolkit - spaCy & NLTK)

In [2]:
print("Upload input text file for 4.1 & 4.2:")
uploaded_4_1 = files.upload()
filename_4_1 = list(uploaded_4_1.keys())[0]

# Read file content
with open(filename_4_1, 'r', encoding='utf-8', errors='ignore') as f:
    text_4_1 = f.read()

# --- PART A: Term-Frequency Analysis using spaCy ---
doc = nlp(text_4_1)

# Extract word tokens (excluding punctuation, spaces, and stopwords)
words_toolkit = [token.text.lower() for token in doc if token.is_alpha]

# Calculate term frequencies
tf_counts_toolkit = Counter(words_toolkit)

# Convert to Pandas DataFrame and export to CSV
df_tf_toolkit = pd.DataFrame(tf_counts_toolkit.items(), columns=['Term', 'Frequency']).sort_values(by='Frequency', ascending=False)
df_tf_toolkit.to_csv('term_frequency_toolkit.csv', index=False)
files.download('term_frequency_toolkit.csv')

print("\n=== EXPERIMENT 4.1: TOP 10 MOST FREQUENT TERMS (WITH TOOLKIT) ===")
display(df_tf_toolkit.head(10))

# --- PART B: Named Entity Recognition (NER) using spaCy ---
print("\n=== EXPERIMENT 4.1: NAMED ENTITY RECOGNITION (NER) ===")
entities = [(ent.text, ent.label_) for ent in doc.ents]

if entities:
    df_ner = pd.DataFrame(entities, columns=['Entity', 'Label']).drop_duplicates()
    display(df_ner)
else:
    print("No named entities found in the text.")

Upload input text file for 4.1 & 4.2:


Saving 4.1_4.2_input.txt to 4.1_4.2_input.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


=== EXPERIMENT 4.1: TOP 10 MOST FREQUENT TERMS (WITH TOOLKIT) ===


,Term,Frequency
98,a,88
3,and,86
50,the,75
14,of,42
101,can,42
20,to,42
42,in,42
10,is,37
159,word,30
113,may,30



=== EXPERIMENT 4.1: NAMED ENTITY RECOGNITION (NER) ===


,Entity,Label
0,Natural Language Processing and Artificial Int...,ORG
1,NLP,ORG
2,English,LANGUAGE
3,Hindi,GPE
4,Assamese,NORP
5,Bengali,NORP
6,Tamil,GPE
10,twenty,CARDINAL
11,twenty,DATE
12,Python,GPE


Experiment 4.2: Term-Frequency Analysis (Without Toolkit - Pure Python)

In [3]:
# --- Term-Frequency Analysis Without Toolkit ---

# 1. Convert text to lowercase
lowercased_text = text_4_1.lower()

# 2. Remove punctuation using regex
clean_text = re.sub(r'[^\w\s]', '', lowercased_text)

# 3. Tokenize text by splitting on whitespace
words_no_toolkit = clean_text.split()

# 4. Calculate word frequencies using standard Python dictionary
tf_counts_no_toolkit = {}
for word in words_no_toolkit:
    if word:  # Skip empty strings
        tf_counts_no_toolkit[word] = tf_counts_no_toolkit.get(word, 0) + 1

# 5. Convert to DataFrame, sort, and save to CSV
df_tf_no_toolkit = pd.DataFrame(list(tf_counts_no_toolkit.items()), columns=['Term', 'Frequency'])
df_tf_no_toolkit = df_tf_no_toolkit.sort_values(by='Frequency', ascending=False)

# Save output to CSV file
df_tf_no_toolkit.to_csv('term_frequency_no_toolkit.csv', index=False)
files.download('term_frequency_no_toolkit.csv')

print("\n=== EXPERIMENT 4.2: TOP 10 MOST FREQUENT TERMS (WITHOUT TOOLKIT) ===")
display(df_tf_no_toolkit.head(10))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


=== EXPERIMENT 4.2: TOP 10 MOST FREQUENT TERMS (WITHOUT TOOLKIT) ===


,Term,Frequency
97,a,90
3,and,86
50,the,75
20,to,42
42,in,42
100,can,41
14,of,40
10,is,37
158,word,30
112,may,30


Experiment 4.3: Complete TF-IDF Step-by-Step (Without Toolkit)

In [4]:
# 1. Define input documents
documents = {
    "Document 1": "Natural language processing is a field of artificial intelligence.",
    "Document 2": "Natural language processing helps computers understand human language.",
    "Document 3": "Machine learning is an important part of artificial intelligence."
}

N = len(documents)

# Helper function for text preprocessing (Lowercase + Punctuation removal + Tokenization)
def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.split()

# Tokenize all documents
tokenized_docs = {doc_id: preprocess(text) for doc_id, text in documents.items()}

# Get total vocabulary across all documents
vocab = sorted(list(set(word for words in tokenized_docs.values() for word in words)))

print(f"Total Unique Vocabulary Count: {len(vocab)}\n")

# --- STEP 1: Calculate Term Frequency (TF) for each document ---
# Formula: TF(t, d) = (Count of term t in d) / (Total words in d)
tf_scores = {}
for doc_id, words in tokenized_docs.items():
    doc_len = len(words)
    word_counts = Counter(words)
    tf_scores[doc_id] = {word: word_counts[word] / doc_len for word in vocab}

# --- STEP 2: Calculate Document Frequency (DF) & Inverse Document Frequency (IDF) ---
# Formula: DF(t) = Number of documents containing term t
# Formula: IDF(t) = log10(N / DF(t))
df_scores = {}
idf_scores = {}

for word in vocab:
    doc_count = sum(1 for words in tokenized_docs.values() if word in words)
    df_scores[word] = doc_count
    idf_scores[word] = math.log10(N / doc_count)

# --- STEP 3: Calculate TF-IDF for every term in every document ---
# Formula: TF-IDF(t, d) = TF(t, d) * IDF(t)
tfidf_scores = {}
for doc_id in documents:
    tfidf_scores[doc_id] = {word: tf_scores[doc_id][word] * idf_scores[word] for word in vocab}

# --- DISPLAY STEP-BY-STEP RESULTS FOR EACH DOCUMENT ---
for doc_id in documents:
    print("=" * 70)
    print(f"  STEP-BY-STEP CALCULATIONS FOR: {doc_id.upper()}")
    print("=" * 70)

    doc_table = []
    for word in sorted(tokenized_docs[doc_id]):
        tf_val = tf_scores[doc_id][word]
        df_val = df_scores[word]
        idf_val = idf_scores[word]
        tfidf_val = tfidf_scores[doc_id][word]

        doc_table.append({
            'Term': word,
            'TF': round(tf_val, 4),
            'DF': df_val,
            'IDF': round(idf_val, 4),
            'TF-IDF': round(tfidf_val, 4)
        })

    # Display full calculation table for the document
    df_doc_result = pd.DataFrame(doc_table).drop_duplicates().reset_index(drop=True)
    display(df_doc_result)

    # Identify Top 10 Terms with highest TF-IDF score
    top_10_tfidf = df_doc_result.sort_values(by='TF-IDF', ascending=False).head(10)
    print(f"\n--- TOP 10 HIGHEST TF-IDF TERMS FOR {doc_id.upper()} ---")
    display(top_10_tfidf[['Term', 'TF-IDF']])
    print("\n")

Total Unique Vocabulary Count: 18

  STEP-BY-STEP CALCULATIONS FOR: DOCUMENT 1


,Term,TF,DF,IDF,TF-IDF
0,a,0.1111,1,0.4771,0.0530
1,artificial,0.1111,2,0.1761,0.0196
2,field,0.1111,1,0.4771,0.0530
3,intelligence,0.1111,2,0.1761,0.0196
4,is,0.1111,2,0.1761,0.0196
5,language,0.1111,2,0.1761,0.0196
6,natural,0.1111,2,0.1761,0.0196
7,of,0.1111,2,0.1761,0.0196
8,processing,0.1111,2,0.1761,0.0196



--- TOP 10 HIGHEST TF-IDF TERMS FOR DOCUMENT 1 ---


,Term,TF-IDF
0,a,0.0530
2,field,0.0530
1,artificial,0.0196
3,intelligence,0.0196
4,is,0.0196
5,language,0.0196
6,natural,0.0196
7,of,0.0196
8,processing,0.0196




  STEP-BY-STEP CALCULATIONS FOR: DOCUMENT 2


...

,Term,TF-IDF
0,computers,0.0596
1,helps,0.0596
2,human,0.0596
6,understand,0.0596
3,language,0.0440
4,natural,0.0220
5,processing,0.0220




  STEP-BY-STEP CALCULATIONS FOR: DOCUMENT 3


,Term,TF,DF,IDF,TF-IDF
0,an,0.1111,1,0.4771,0.0530
1,artificial,0.1111,2,0.1761,0.0196
2,important,0.1111,1,0.4771,0.0530
3,intelligence,0.1111,2,0.1761,0.0196
4,is,0.1111,2,0.1761,0.0196
5,learning,0.1111,1,0.4771,0.0530
6,machine,0.1111,1,0.4771,0.0530
7,of,0.1111,2,0.1761,0.0196
8,part,0.1111,1,0.4771,0.0530



--- TOP 10 HIGHEST TF-IDF TERMS FOR DOCUMENT 3 ---


,Term,TF-IDF
0,an,0.0530
2,important,0.0530
8,part,0.0530
6,machine,0.0530
5,learning,0.0530
4,is,0.0196
3,intelligence,0.0196
1,artificial,0.0196
7,of,0.0196
